<a href="https://colab.research.google.com/github/nubar-mamedova/wana-solar-investment-analysis/blob/main/wana_solar_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [220]:
# WANA Solar Analysis
# Goal: analyze 17 WANA countries for utility-scale solar investment
# across six analytical lenses, not one single composite ranking.
#
# Data sources:
# 1. Global Solar Atlas
# 2. World Bank API via wbgapi
# 3. Our World in Data solar share CSV

In [221]:
## Setup — installs and imports
!pip install wbgapi --quiet

import os
import pandas as pd
import wbgapi as wb

In [222]:
## Paths — Drive mount and folder paths
# Paths assume Colab + Drive. To run locally, set DATAPATH and PROCESSEDPATH
# to local folders containing gsa_country_pvpotential_global.xlsx.
from google.colab import drive
drive.mount('/content/drive')

DATAPATH = "/content/drive/MyDrive/Colab Notebooks/mena-energy-market-expansion/data/raw"
PROCESSEDPATH = "/content/drive/MyDrive/Colab Notebooks/mena-energy-market-expansion/data/processed"
os.makedirs(PROCESSEDPATH, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [223]:
## GSA — load and inspect raw file
gsapath = os.path.join(DATAPATH, "gsa_country_pvpotential_global.xlsx")
gsa = pd.read_excel(gsapath, sheet_name="Country indicators", header=1)

print(gsa.shape)
print(gsa.columns.tolist())
print(gsa[["ISO_A3", "Country or region"]].head())
assert gsa.shape[0] == 209, "GSA file should have 209 country rows"

(209, 21)
['ISO_A3', 'Country or region', 'Note', 'World Bank \nRegion', 'Total population, 2018', 'Total area, 2018', 'Evaluated area', 'Level 1 area \n(% of evaluated area)', 'Human development \nIndex, 2017', 'Gross domestic product (USD per capita), 2018', 'Average theoretical potential (GHI, kWh/m2/day), \nlong-term', 'Average practical potential \n(PVOUT Level 1, \nkWh/kWp/day), long-term', 'Average economic potential (LCOE, USD/kWh), 2018', 'Average PV \nseasonality index, long-term', 'PV equivalent area (% of total area), long-term', 'Cummulative installed PV capacity (MWp), 2018', 'Cummulative installed PV capacity (Wp per capita), 2018', 'Access to electricity\n(% of rural population), 2016', 'Electric power consumption (kWh per capita), 2014', 'Reliability of supply and transparency of tariff index, 2019', 'Approximate electricity \nTariffs for SMEs \n(US cent/kWh), 2019']
  ISO_A3 Country or region
0    ABW     Aruba (Neth.)
1    AFG       Afghanistan
2    AGO            An

In [224]:
## GSA — confirm solar column names
solar_cols = [
    col for col in gsa.columns
    if "theoretical" in str(col).lower() or "practical" in str(col).lower()
]

print("Solar-related columns found:")
for col in solar_cols:
    print(f"  - {repr(col)}")

Solar-related columns found:
  - 'Average theoretical potential (GHI, kWh/m2/day), \nlong-term'
  - 'Average practical potential \n(PVOUT Level 1, \nkWh/kWp/day), long-term'


In [225]:
## WANA — country list and ISO codes
WANACOUNTRIES = {
    "DZA": "Algeria",
    "BHR": "Bahrain",
    "EGY": "Egypt",
    "ETH": "Ethiopia",
    "IRN": "Iran",
    "IRQ": "Iraq",
    "JOR": "Jordan",
    "KWT": "Kuwait",
    "LBN": "Lebanon",
    "LBY": "Libya",
    "MAR": "Morocco",
    "QAT": "Qatar",
    "SAU": "Saudi Arabia",
    "TUN": "Tunisia",
    "TUR": "Turkey",
    "ARE": "United Arab Emirates",
    "YEM": "Yemen"
}
WANAISO3 = list(WANACOUNTRIES.keys())

In [226]:
## GSA — filter to 17 WANA countries
gsawana = gsa[gsa["ISO_A3"].isin(WANAISO3)].copy()
print(f"Filtered to {len(gsawana)} WANA countries")
print(gsawana[["ISO_A3", "Country or region"]].sort_values("ISO_A3"))
assert len(gsawana) == 17, f"Expected 17 countries after filter, got {len(gsawana)}"


Filtered to 17 WANA countries
    ISO_A3         Country or region
5      ARE      United Arab Emirates
19     BHR                   Bahrain
54     DZA                   Algeria
56     EGY    Arab Republic of Egypt
60     ETH                  Ethiopia
86     IRN  Islamic Republic of Iran
87     IRQ                      Iraq
91     JOR                    Jordan
100    KWT                    Kuwait
102    LBN                   Lebanon
104    LBY                     Libya
113    MAR                   Morocco
155    QAT                     Qatar
159    SAU              Saudi Arabia
188    TUN                   Tunisia
189    TUR                    Turkey
205    YEM         Republic of Yemen


In [227]:
## World Bank — pull indicators via API
INDICATORS = {
    "NY.GDP.PCAP.CD": "gdp_per_capita_usd",
    "SP.POP.TOTL": "population_total",
    "SP.URB.TOTL.IN.ZS": "urban_population_pct",
    "EG.USE.ELEC.KH.PC": "electricity_kwh_per_capita",
    "EG.ELC.ACCS.ZS": "access_to_electricity_pct",
    "EG.ELC.RNEW.ZS": "renewable_electricity_pct",
    "SP.POP.GROW": "population_growth_pct",
}

wb_long = wb.data.DataFrame(
    list(INDICATORS.keys()),
    WANAISO3,
    time=range(2010, 2024),
    labels=False
).reset_index()

wb_long = wb_long.rename(columns={
    "economy": "iso3",
    "series": "indicator_code"
})

wb_long["indicator"] = wb_long["indicator_code"].map(INDICATORS)
wb_long["country"] = wb_long["iso3"].map(WANACOUNTRIES)

In [228]:
## World Bank — tidy and pivot to wide format
year_cols = [c for c in wb_long.columns if c.startswith("YR")]

wb_tidy = wb_long.melt(
    id_vars=["iso3", "country", "indicator"],
    value_vars=year_cols,
    var_name="year",
    value_name="value"
)

wb_tidy["year"] = wb_tidy["year"].str.replace("YR", "", regex=False).astype(int)
wb_tidy = wb_tidy.dropna(subset=["value"])

# Take latest non-null year per country/indicator (data availability varies)
wb_latest = (
    wb_tidy.sort_values("year")
    .groupby(["iso3", "country", "indicator"])
    .tail(1)
    .reset_index(drop=True)
)

wb_wide = (
    wb_latest.pivot(
        index=["iso3", "country"],
        columns="indicator",
        values="value"
    )
    .reset_index()
)

In [229]:
## OWID — solar share over time
OWID_URL = "https://ourworldindata.org/grapher/share-electricity-solar.csv"

owid = pd.read_csv(OWID_URL)
print(owid.columns.tolist())

owid = owid.rename(columns={
    "Code": "iso3",
    "Year": "year",
    "Solar": "solar_share_pct"
})

owid_wana = owid[
    owid["iso3"].isin(WANAISO3) &
    owid["year"].between(2010, 2023)
].copy()

solar_latest = (
    owid_wana.sort_values("year")
    .groupby("iso3")
    .tail(1)
    .rename(columns={
        "solar_share_pct": "solar_share_latest",
        "year": "solar_share_year"
    })[["iso3", "solar_share_latest", "solar_share_year"]]
)

owid_recent = owid_wana[owid_wana["year"] >= owid_wana["year"].max() - 5]

trajectory = (
    owid_recent.sort_values(["iso3", "year"])
    .groupby("iso3")
    .agg(
        solar_share_5y_ago=("solar_share_pct", "first"),
        solar_share_now=("solar_share_pct", "last")
    )
    .reset_index()
)

trajectory["solar_share_change_5y"] = (
    trajectory["solar_share_now"] - trajectory["solar_share_5y_ago"]
)

['Entity', 'Code', 'Year', 'Solar']


In [230]:
## GSA — clean column names for master
SOLAR_THEORETICAL = 'Average theoretical potential (GHI, kWh/m2/day), \nlong-term'
SOLAR_PRACTICAL = 'Average practical potential \n(PVOUT Level 1, \nkWh/kWp/day), long-term'
GDP2018 = 'Gross domestic product (USD per capita), 2018'
POP2018 = 'Total population, 2018'
AREA2018 = 'Total area, 2018'
HDI2017 = 'Human development \nIndex, 2017'

In [231]:
gsaclean = gsawana[
    ["ISO_A3", "Country or region", SOLAR_THEORETICAL, SOLAR_PRACTICAL, GDP2018, POP2018, AREA2018, HDI2017]
].copy()

gsaclean.columns = [
    "iso3", "country_raw", "solar_theoretical_kwh_m2_day", "solar_practical_kwh_kwp_day",
    "gdp_per_capita_usd_2018", "population_2018", "area_sqkm_2018", "hdi_2017"
]

gsaclean["country"] = gsaclean["iso3"].map(WANACOUNTRIES)



In [232]:
## Master — merge three sources
master = (
    gsaclean
    .merge(wb_wide.drop(columns="country"), on="iso3", how="left")
    .merge(solar_latest, on="iso3", how="left")
    .merge(trajectory[["iso3", "solar_share_change_5y"]], on="iso3", how="left")
)

assert len(master) == 17, f"Master should have 17 rows, got {len(master)}"

missingness = master.isna().sum().sort_values(ascending=False)
if missingness.sum() == 0:
    print("No missing values after merge.")
else:
    print("Missing values by column:")
    print(missingness[missingness > 0])

print(f"Master shape: {master.shape}")

No missing values after merge.
Master shape: (17, 19)


In [233]:
## Master — save to processed folder
master.to_csv(os.path.join(PROCESSEDPATH, "wana_solar_master.csv"), index=False)

In [234]:
## Helpers — normalize functions

def normalize_higher_better(series):
    rng = series.max() - series.min()
    if rng == 0:
        return pd.Series(50.0, index=series.index)
    return 100 * (series - series.min()) / rng

def normalize_lower_better(series):
    rng = series.max() - series.min()
    if rng == 0:
        return pd.Series(50.0, index=series.index)
    return 100 * (series.max() - series) / rng

In [235]:
## Helpers — filter_ranking_data
def filter_ranking_data(df, required_cols, ranking_name):
    before = set(df["country"])
    clean = df.dropna(subset=required_cols).copy()
    after = set(clean["country"])
    dropped = sorted(before - after)
    print(f"[{ranking_name}] using {len(after)}/{len(before)} countries.")
    if dropped:
        print(f"  excluded (missing data): {', '.join(dropped)}")
    return clean, dropped

In [236]:
## Rankings — six ranking functions
def rank_resource(df):
    cols = ["solar_practical_kwh_kwp_day"]
    clean, dropped = filter_ranking_data(df, cols, "resource")
    clean["score"] = normalize_higher_better(clean["solar_practical_kwh_kwp_day"])
    clean["rank"] = clean["score"].rank(ascending=False, method="min").astype(int)
    return clean.sort_values("rank"), dropped

def rank_market_scale(df):
    cols = ["gdp_per_capita_usd", "population_total", "electricity_kwh_per_capita"]
    clean, dropped = filter_ranking_data(df, cols, "market_scale")
    clean["gdptotalusd"] = clean["gdp_per_capita_usd"] * clean["population_total"]
    parts = pd.DataFrame({
        "gdptotal":      normalize_higher_better(clean["gdptotalusd"]),
        "electricitypc": normalize_higher_better(clean["electricity_kwh_per_capita"]),
        "population":    normalize_higher_better(clean["population_total"])
    }, index=clean.index)
    clean["score"] = parts.mean(axis=1)
    clean["rank"]  = clean["score"].rank(ascending=False, method="min").astype(int)
    return clean.sort_values("rank"), dropped

def rank_untapped(df):
    cols = ["solar_practical_kwh_kwp_day", "solar_share_latest"]
    clean, dropped = filter_ranking_data(df, cols, "untapped")
    clean["untappedratio"] = clean["solar_practical_kwh_kwp_day"] / (clean["solar_share_latest"] + 0.1)
    clean["score"] = normalize_higher_better(clean["untappedratio"])
    clean["rank"]  = clean["score"].rank(ascending=False, method="min").astype(int)
    return clean.sort_values("rank"), dropped

def rank_execution_readiness(df):
    cols = ["urban_population_pct", "access_to_electricity_pct", "electricity_kwh_per_capita"]
    clean, dropped = filter_ranking_data(df, cols, "execution_readiness")
    parts = pd.DataFrame({
        "urban":  normalize_higher_better(clean["urban_population_pct"]),
        "access": normalize_higher_better(clean["access_to_electricity_pct"]),
        "grid":   normalize_higher_better(clean["electricity_kwh_per_capita"])
    }, index=clean.index)
    clean["score"] = parts.mean(axis=1)
    clean["rank"]  = clean["score"].rank(ascending=False, method="min").astype(int)
    return clean.sort_values("rank"), dropped

def rank_trajectory(df):
    cols = ["solar_share_change_5y"]
    clean, dropped = filter_ranking_data(df, cols, "trajectory")
    clean["score"] = normalize_higher_better(clean["solar_share_change_5y"])
    clean["rank"]  = clean["score"].rank(ascending=False, method="min").astype(int)
    return clean.sort_values("rank"), dropped

def rank_demand_pressure(df):
    cols = ["population_growth_pct", "electricity_kwh_per_capita"]
    clean, dropped = filter_ranking_data(df, cols, "demand_pressure")
    parts = pd.DataFrame({
        "popgrowth":   normalize_higher_better(clean["population_growth_pct"]),
        "consumption": normalize_higher_better(clean["electricity_kwh_per_capita"])
    }, index=clean.index)
    clean["score"] = parts.mean(axis=1)
    clean["rank"]  = clean["score"].rank(ascending=False, method="min").astype(int)
    return clean.sort_values("rank"), dropped

In [237]:
## Rankings — run all six
RANKINGFUNCS = {
    "resource":            rank_resource,
    "market_scale":        rank_market_scale,
    "untapped":            rank_untapped,
    "execution_readiness": rank_execution_readiness,
    "trajectory":          rank_trajectory,
    "demand_pressure":     rank_demand_pressure,
}

results = {}
exclusions = {}

for name, fn in RANKINGFUNCS.items():
    df_ranked, dropped = fn(master)
    results[name] = df_ranked
    exclusions[name] = dropped

[resource] using 17/17 countries.
[market_scale] using 17/17 countries.
[untapped] using 17/17 countries.
[execution_readiness] using 17/17 countries.
[trajectory] using 17/17 countries.
[demand_pressure] using 17/17 countries.


In [238]:
## Tableau — save exclusions audit
exclusionsdf = pd.DataFrame([
    {"ranking": name, "country": country}
    for name, drops in exclusions.items()
    for country in drops
])

exclusionsdf.to_csv(os.path.join(PROCESSEDPATH, "wana_excluded_countries.csv"), index=False)

In [239]:
## Tableau — score wide table
scorewide = master[["iso3", "country"]].copy()

for name, df_ in results.items():
    scorewide = scorewide.merge(
        df_[["iso3", "score"]].rename(columns={"score": f"score_{name}"}),
        on="iso3",
        how="left"
    )

In [240]:
## Tableau — archetypes
ARCHETYPES = {
    "developmentbank": {
        "resource": 0.10, "market_scale": 0.10, "untapped": 0.15,
        "execution_readiness": 0.30, "trajectory": 0.10, "demand_pressure": 0.25
    },
    "ippdeveloper": {
        "resource": 0.35, "market_scale": 0.10, "untapped": 0.10,
        "execution_readiness": 0.30, "trajectory": 0.05, "demand_pressure": 0.10
    },
    "hyperscalerppa": {
        "resource": 0.30, "market_scale": 0.30, "untapped": 0.05,
        "execution_readiness": 0.20, "trajectory": 0.05, "demand_pressure": 0.10
    },
    "esgfund": {
        "resource": 0.10, "market_scale": 0.10, "untapped": 0.30,
        "execution_readiness": 0.10, "trajectory": 0.30, "demand_pressure": 0.10
    }
}

In [241]:
## Tableau — rankings long format
for archname, w in ARCHETYPES.items():
    scorewide[f"archetype_{archname}"] = sum(
        w[k] * scorewide[f"score_{k}"] for k in w
    )
    scorewide[f"rank_{archname}"] = (
        scorewide[f"archetype_{archname}"]
        .rank(ascending=False, method="min")
    )

longrows = []
for name, df_ in results.items():
    for _, row in df_.iterrows():
        longrows.append({
            "ranking": name,
            "iso3": row["iso3"],
            "country": row["country"],
            "rank": row["rank"],
            "score": row["score"]
        })

rankingslong = pd.DataFrame(longrows)
rankingslong.to_csv(os.path.join(PROCESSEDPATH, "wana_rankings_long.csv"), index=False)

In [242]:
## Tableau — archetype long format
archlongrows = []
for archname in ARCHETYPES:
    sub = scorewide[["iso3", "country", f"archetype_{archname}", f"rank_{archname}"]].copy()
    sub = sub.rename(columns={
        f"archetype_{archname}": "score",
        f"rank_{archname}": "rank"
    })
    sub["archetype"] = archname
    archlongrows.append(sub)

archetypeslong = pd.concat(archlongrows, ignore_index=True)
archetypeslong.to_csv(os.path.join(PROCESSEDPATH, "wana_archetypes_long.csv"), index=False)

In [243]:
## Tableau — ranking labels
labelrows = []
for name, df_ in results.items():
    labelrows.append({
        "ranking": name,
        "ncountriesused": len(df_),
        "ncountriestotal": 17,
        "countriesexcluded": ", ".join(exclusions[name]) if exclusions[name] else "none"
    })

pd.DataFrame(labelrows).to_csv(
    os.path.join(PROCESSEDPATH, "wana_rankinglabels.csv"),
    index=False
)

In [244]:
## Sensitivity — do top picks survive weight perturbations?
import numpy as np

def perturb_weights(weights, delta=0.20, seed=42):
    rng = np.random.default_rng(seed)
    noise = rng.uniform(1 - delta, 1 + delta, size=len(weights))
    perturbed = {k: w * n for (k, w), n in zip(weights.items(), noise)}
    total = sum(perturbed.values())
    return {k: v / total for k, v in perturbed.items()}

N_TRIALS = 100
for archname, base_w in ARCHETYPES.items():
    top3_counts = {}
    for i in range(N_TRIALS):
        w = perturb_weights(base_w, delta=0.20, seed=i)
        scores = sum(w[k] * scorewide[f"score_{k}"] for k in w)
        top3 = scorewide.assign(s=scores).nlargest(3, "s")["country"].tolist()
        for c in top3:
            top3_counts[c] = top3_counts.get(c, 0) + 1
    top_stable = sorted(top3_counts.items(), key=lambda x: -x[1])[:5]
    print(f"\n[{archname}] top-3 frequency across {N_TRIALS} ±20% weight perturbations:")
    for country, count in top_stable:
        print(f"  {country}: {count}/{N_TRIALS}")


[developmentbank] top-3 frequency across 100 ±20% weight perturbations:
  Bahrain: 100/100
  Kuwait: 100/100
  Saudi Arabia: 97/100
  United Arab Emirates: 3/100

[ippdeveloper] top-3 frequency across 100 ±20% weight perturbations:
  Saudi Arabia: 100/100
  Bahrain: 100/100
  United Arab Emirates: 84/100
  Jordan: 11/100
  Kuwait: 5/100

[hyperscalerppa] top-3 frequency across 100 ±20% weight perturbations:
  Saudi Arabia: 100/100
  United Arab Emirates: 100/100
  Bahrain: 100/100

[esgfund] top-3 frequency across 100 ±20% weight perturbations:
  Libya: 100/100
  Bahrain: 95/100
  Lebanon: 64/100
  Kuwait: 36/100
  United Arab Emirates: 5/100


In [245]:
## Findings — top picks by archetype
for archname in ARCHETYPES:
    top3 = scorewide.nlargest(3, f"archetype_{archname}")[
        ["country", f"archetype_{archname}", f"rank_{archname}"]
    ]
    print(f"\n=== {archname.upper()} ===")
    print(top3.to_string(index=False))


=== DEVELOPMENTBANK ===
     country  archetype_developmentbank  rank_developmentbank
     Bahrain                  65.873214                   1.0
      Kuwait                  63.638621                   2.0
Saudi Arabia                  58.186411                   3.0

=== IPPDEVELOPER ===
             country  archetype_ippdeveloper  rank_ippdeveloper
        Saudi Arabia               67.558597                1.0
             Bahrain               65.023536                2.0
United Arab Emirates               61.783217                3.0

=== HYPERSCALERPPA ===
             country  archetype_hyperscalerppa  rank_hyperscalerppa
        Saudi Arabia                 66.658334                  1.0
United Arab Emirates                 57.719806                  2.0
             Bahrain                 57.293249                  3.0

=== ESGFUND ===
country  archetype_esgfund  rank_esgfund
  Libya          49.590395           1.0
Lebanon          42.426862           2.0
Bahrain      

* Development banks (execution readiness + demand pressure): Bahrain, Kuwait, Saudi Arabia top the list — high grid maturity and rising consumption. Worth flagging that small Gulf states score artificially well on per-capita indicators; a real DFI would weigh population scale more heavily.
* IPP developers (resource + execution): Saudi Arabia, Bahrain, UAE — the expected Gulf trio. Sensitivity shows Jordan creeping into top 3 in 11% of perturbations, making it a credible fourth pick.
* Hyperscaler PPAs (resource + market scale): Saudi, UAE, Bahrain are 100% stable across all perturbations — the most robust result in the model.
ESG funds (untapped + trajectory): Libya, Lebanon, Bahrain. Libya is interesting — top resource potential, near-zero current solar share — but political risk is not modeled here and would be the obvious next layer to add.